# Introduction

These notebook contains the steps for evaluating my model on both modalities DWM and naive. Although, for testing the naive model, one has to make minor adjustments in the scritps `exp_regressor`, at the prediction step.

`Model comparison`: we can find here the `quantitative evaluation` for the pancancer experiment only and for all combinations between NBS cell, NBS drugs and fixed-cell or fixed-drug. However, the NBS drug is currently not bein included because ScreenDL is not designed for this setting.


# Libraries 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torchmetrics as tm
import torch.nn.functional as F
import seaborn as sns
torch.manual_seed(9497)
import scipy.stats as stats
import plotly.express as px
from pathlib import Path
import umap
from sklearn.decomposition import PCA
from statsmodels.stats.multitest import multipletests
from scipy.stats import wilcoxon
import plotly.graph_objects as go

# Model 

In [2]:
from exp_regression import DrugAware_DWM, Transcriptomic_Head, SMILES_Head, CombinedModel
from cli_regressor import Module_training_reg
from utilities import CVDataModule

In [4]:
tx_head = Transcriptomic_Head(
            input_size=946, # 946 | 4377
            hd_1=512,
            hd_2=256
        )

chem_head = SMILES_Head(
    input_size=512,
    hd_1=512, 
    hd_2=256,
    hd_3=128
)

combined_model = CombinedModel(
    input_size=1024,
    hd_1=256, #256
    hd_2=128  #128
)

loss_fn = nn.MSELoss()
lr= 1e-4
wd=0.0

# Cross validation

In [5]:
def precision_at_q(y_true, y_hat, q=0.25):
    y_true = np.asarray(y_true)
    y_hat = np.asarray(y_hat)

    thr = np.quantile(y_true, q)
    true_pos = np.where(y_true <= thr)[0]
    k = len(true_pos)

    pred_topk = np.argsort(y_hat)[:k]

    return len(set(true_pos) & set(pred_topk)) / k

def ndcg_at_q(y_true, y_hat, q=0.25):
    "Normalized discounted cumulative gain"
    y_true = np.asarray(y_true)
    y_hat = np.asarray(y_hat)

    thr = np.quantile(y_true, q)
    true_pos = np.where(y_true <= thr)[0]
    k = len(true_pos)

    # relevance: higher is better
    rel = -y_true
    
    # predicted ranking
    order = np.argsort(y_hat)
    rel_pred = rel[order][:k]
    
    discounts = 1 / np.log2(np.arange(2, k + 2)) # rank weight
    dcg = np.sum((2 ** rel_pred - 1) * discounts)

    # ideal ranking
    ideal_order = np.argsort(y_true)
    rel_ideal = rel[ideal_order][:k]
    idcg = np.sum((2 ** rel_ideal - 1) * discounts)

    return dcg / idcg if idcg > 0 else 0.0

In [6]:
# expression_data = pd.read_csv("output/regression/GEX_data_filtered_logCPM.csv", index_col=0)
# vector_smiles = pd.read_csv("output/regression/vector_smiles_512.csv", index_col=0)

root = "output/regression/cross_validation"
cancer_type = "pancancer" # pancancer , solid_tumors
experiment = "NBS_cells"

fold_dirs = Path(f"{root}/{cancer_type}/{experiment}/")
ckpt_dir = Path(f"cDWM/L1000_{cancer_type}_NBS_cells/")
# ckpt_dir = Path(f"naive_predictor/L1000_{cancer_type}_NBS_cells/")

# Sort fold paths in numeric order (fold_0, fold_1, ...)
folds = sorted(fold_dirs.glob("fold_*"), key=lambda x: int(x.name.split("_")[1]))
ckpts = sorted(ckpt_dir.glob("version_*"), key=lambda x: int(x.name.split("_")[1]))

per_drug_pcc_cv = {}
per_cell_pcc_cv = {}

per_drug_precision_cv = {}
per_cell_precision_cv = {}

per_drug_ndcg_cv = {}
per_cell_ndcg_cv = {}


for fold_path, ckpt_path in zip(folds, ckpts):
    fold_name = fold_path.name.split("_")[1]
    ckpt_file = list((ckpt_path / 'checkpoints').glob('*.ckpt'))[0]

    data_module = CVDataModule(
        root = root,
        type = cancer_type,
        experiment = experiment,
        fold_n = fold_name,
        GEX_path = "output/regression/L1000_GEX_data_filtered_logCPM.csv",
        SMILES_path = "output/regression/vector_smiles_512.csv",
        batch_size = 512,
        num_workers = 12
    )

    gex, sm, test_set = data_module.test_dataloader()

    # Prediction
    regressor = Module_training_reg.load_from_checkpoint(
    ckpt_file,
    map_location="cpu",
    tx_nn=tx_head,
    chem_nn=chem_head,
    comb_nn=combined_model,
    loss_fn=loss_fn,
    lr=lr,
    weight_decay=wd
    )

    y_hat = regressor.predict(
        gex=gex,
        sm=sm
    )
    
    y_hat = y_hat.double()
    y_hat = y_hat.numpy()

    test_set["Y_HAT"] = y_hat

    # Computing metrics
    ##############################
    # PCC
    ##############################
    per_drug_pcc = (test_set.groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())

    per_drug_pcc_cv[fold_name] = [per_drug_pcc["PCC"].median()]

    per_cell_pcc = (test_set.groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())

    per_cell_pcc_cv[fold_name] = [per_cell_pcc["PCC"].median()]

    ##############################
    # PRECISION
    ##############################
    per_drug_pcc = (test_set.groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_drug_precision_cv[fold_name] = [per_drug_pcc["precision@q25"].median()]

    per_cell_pcc = (test_set.groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_cell_precision_cv[fold_name] = [per_cell_pcc["precision@q25"].median()]

    ##############################
    # NDCG
    ##############################
    per_drug_pcc = (test_set.groupby("DRUG_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("DRUG_NAME")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_drug_ndcg_cv[fold_name] = [per_drug_pcc["ndcg@q25"].median()]

    per_cell_pcc = (test_set.groupby("CELL_LINE_NAME")
                    .filter(lambda x: len(x) >=2)
                    .groupby("CELL_LINE_NAME")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_cell_ndcg_cv[fold_name] = [per_cell_pcc["ndcg@q25"].median()]




/tmp/ipykernel_11842/3234134936.py:72: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_11842/3234134936.py:80: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_11842/3234134936.py:91: FutureWarning: DataFr

In [6]:
per_drug_pcc_cv = pd.DataFrame(per_drug_pcc_cv)
per_drug_pcc_cv.index = ["naive-NELLY"]
per_drug_pcc_cv

,0,1,2,3,4,5,6,7,8,9
naive-NELLY,0.471384,0.479756,0.464224,0.510227,0.500246,0.478946,0.513859,0.537112,0.489459,0.461765


In [7]:
M = np.median(per_drug_pcc_cv)
sem = stats.sem(per_drug_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.4846075746866744, low=0.4671929641374656, high=0.5020221852358833


In [ ]:
# per_drug_pcc_cv.to_csv(
#     "cDWM/L1000_pancancer_NBS_cells/fixed-drug_CV.csv"
# )

In [ ]:
# naive_per_drug = per_drug_pcc_cv

# naive_per_drug.to_csv(
#     "naive_predictor/L1000_pancancer_NBS_cells/fixed-drug_CV.csv"
# )

In [8]:
per_drug_precision_cv = pd.DataFrame(per_drug_precision_cv)
per_drug_precision_cv.index = ["naive-NELLY"]
per_drug_precision_cv

,0,1,2,3,4,5,6,7,8,9
naive-NELLY,0.521739,0.52381,0.5,0.565217,0.521739,0.5,0.5,0.590909,0.536585,0.5


In [9]:
M = np.median(per_drug_precision_cv)
sem = stats.sem(per_drug_precision_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.5217391304347826, low=0.49961523847275907, high=0.5438630223968061


In [ ]:
# per_drug_precision_cv.to_csv(
#     "cDWM/L1000_pancancer_NBS_cells/precision_fixed-drug_CV.csv"
# )

In [ ]:
# per_drug_precision_cv.to_csv(
#     "naive_predictor/L1000_pancancer_NBS_cells/precision_fixed-drug_CV.csv"
# )

In [11]:
per_drug_ndcg_cv = pd.DataFrame(per_drug_ndcg_cv)
per_drug_ndcg_cv.index = ["naive-NELLY"]
per_drug_ndcg_cv

,0,1,2,3,4,5,6,7,8,9
naive-NELLY,0.435757,0.471821,0.400387,0.487883,0.420864,0.372262,0.405214,0.495934,0.407975,0.384313


In [12]:
M = np.median(per_drug_ndcg_cv)
sem = stats.sem(per_drug_ndcg_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.41441956055218804, low=0.3833933899585906, high=0.44544573114578545


In [ ]:
# per_drug_ndcg_cv.to_csv(
#     "cDWM/L1000_pancancer_NBS_cells/ndgc_fixed-drug_CV.csv"
# )

In [ ]:
# per_drug_ndcg_cv.to_csv(
#     "naive_predictor/L1000_pancancer_NBS_cells/ndgc_fixed-drug_CV.csv"
# )

In [14]:
per_cell_pcc_cv = pd.DataFrame(per_cell_pcc_cv)
per_cell_pcc_cv.index = ["naive-NELLY"]
per_cell_pcc_cv

,0,1,2,3,4,5,6,7,8,9
naive-NELLY,0.909241,0.907302,0.900868,0.902207,0.908741,0.900075,0.90989,0.903009,0.902387,0.903051


In [15]:
M = np.median(per_cell_pcc_cv)
sem = stats.sem(per_cell_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.9030298531129335, low=0.9003759125227421, high=0.9056837937031249


In [ ]:
# per_cell_pcc_cv.to_csv(
#     "cDWM/L1000_pancancer_NBS_cells/fixed-cell_CV.csv"
# )

In [ ]:
# naive_per_cel = per_cell_pcc_cv

# naive_per_cel.to_csv(
#     "naive_predictor/L1000_pancancer_NBS_cells/fixed-cell_CV.csv"
# )

In [16]:
per_cell_precision_cv = pd.DataFrame(per_cell_precision_cv)
per_cell_precision_cv.index = ["naive-NELLY"]
per_cell_precision_cv

,0,1,2,3,4,5,6,7,8,9
naive-NELLY,0.818665,0.807655,0.805195,0.81,0.821053,0.804598,0.8125,0.804598,0.807692,0.810526


In [17]:
M = np.median(per_cell_precision_cv)
sem = stats.sem(per_cell_precision_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.8088461538461539, low=0.804747511613888, high=0.8129447960784197


In [ ]:
# per_cell_precision_cv.to_csv(
#     "cDWM/L1000_pancancer_NBS_cells/precision_fixed-cell_CV.csv"
# )

In [ ]:
# per_cell_precision_cv.to_csv(
#     "naive_predictor/L1000_pancancer_NBS_cells/precision_fixed-cell_CV.csv"
# )

In [19]:
per_cell_ndcg_cv = pd.DataFrame(per_cell_ndcg_cv)
per_cell_ndcg_cv.index = ["naive-NELLY"]
per_cell_ndcg_cv

,0,1,2,3,4,5,6,7,8,9
naive-NELLY,0.909881,0.90602,0.905796,0.900351,0.906007,0.8994,0.910149,0.894259,0.899674,0.899065


In [20]:
M = np.median(per_cell_ndcg_cv)
sem = stats.sem(per_cell_ndcg_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.9030733483383978, low=0.8993234037219268, high=0.9068232929548689


In [ ]:
# per_cell_ndcg_cv.to_csv(
#     "cDWM/L1000_pancancer_NBS_cells/ndcg_fixed-cell_CV.csv"
# )

In [ ]:
# per_cell_ndcg_cv.to_csv(
#     "naive_predictor/L1000_pancancer_NBS_cells/ndcg_fixed-cell_CV.csv"
# )

# Model comparison

## Pancancer

### NBS cells | Fixed-drug 

#### PCC 

In [22]:
screendl_fixed_drug = pd.read_csv(
    "benchmarking/screendl/predictions_pancancer_NBS_cells/msigbd_fixed-drug_evaluation.csv",
    header=0,
    index_col=0
)

In [23]:
idx = screendl_fixed_drug.columns

In [24]:
screendl_fixed_drug = screendl_fixed_drug.set_axis(
    labels=["0","1","2","3","4","5","6","7","8","9"],
    axis=1
)

In [25]:
hidra_fixed_drug = pd.read_csv(
    "benchmarking/hidra/CV/predictions_pancancer_NBS_cells/fixed-drug_evaluation_df.csv",
    index_col=0
)

In [26]:
hidra_fixed_drug.index = ["HiDRA"]

In [27]:
paccmann_fixed_drug = pd.read_csv(
    "benchmarking/paccmann_predictor/CV/predictions_pancancer_NBS_cells/fixed-drug_evaluation_df.csv",
    index_col=0
)

In [28]:
naive_NELLY_fixed_drug = pd.read_csv(
    "naive_predictor/L1000_pancancer_NBS_cells/fixed-drug_CV.csv",
    index_col=0
)

In [29]:
naive_NELLY_fixed_drug.index = ["naive-NELLY"]

In [30]:
dwm_NELLY_fixed_drug = pd.read_csv(
    "cDWM/L1000_pancancer_NBS_cells/fixed-drug_CV.csv",
    index_col=0
)

In [31]:
dwm_NELLY_fixed_drug.index = ["DWM-NELLY"]

In [35]:
metrics_fixed_drug = pd.concat(
    [
        paccmann_fixed_drug, hidra_fixed_drug,
        screendl_fixed_drug, naive_NELLY_fixed_drug, dwm_NELLY_fixed_drug
    ]
).T

In [36]:
metrics_fixed_drug

,Paccmann,HiDRA,ScreenDL,naive-NELLY,DWM-NELLY
0,0.277194,0.415448,0.462896,0.471384,0.731147
1,0.288726,0.387499,0.433751,0.479756,0.742444
2,0.445097,0.394137,0.444695,0.464224,0.758396
3,0.516187,0.423676,0.464933,0.510227,0.773522
4,0.232899,0.456916,0.492156,0.500246,0.755967
5,0.450590,0.379970,0.461375,0.478946,0.755709
6,0.347161,0.417523,0.480839,0.513859,0.765123
7,0.510072,0.448816,0.503738,0.537112,0.783764
8,0.525021,0.411269,0.453561,0.489459,0.753365
9,0.256598,0.306434,0.403882,0.461765,0.472165


In [14]:
metrics_fixed_drug.index = idx

In [15]:
metrics_fixed_drug.reset_index(inplace=True)

In [16]:
metrics_fixed_drug = metrics_fixed_drug.melt(
    id_vars="index",
    var_name="Method",
    value_name="MCorrelation"
)

In [17]:
metrics_fixed_drug.head()

,index,Method,MCorrelation
0,fold_0,Paccmann,0.277194
1,fold_1,Paccmann,0.288726
2,fold_2,Paccmann,0.445097
3,fold_3,Paccmann,0.516187
4,fold_4,Paccmann,0.232899


In [35]:
palette = {
    "Paccmann": "#1F77B4",       # blue
    "HiDRA": "#FF7F0E",  # orange
    "ScreenDL": "#2CA02C",     # green
    "naive-NELLY": "#D62728",   # red
    "DWM-NELLY" : "#b38ce3"
}

fig = px.violin(metrics_fixed_drug,
                x="Method",
                y="MCorrelation",
                color="Method",
                height=500,
                width=700,
                template="simple_white",
                box=True,
                points="all",
                labels={"Method": "",
                        "MCorrelation": "Pearson Correlation"
                       },
                color_discrete_map=palette,
                title= "CV on NBS cells | Fixed-drug Correlation" 
               )


fig.update_traces(
    jitter=0.05,
    pointpos=0,
    width=0.5
)



fig.add_annotation(
    x=0.03,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {paccmann_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.23,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {hidra_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.48,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {screendl_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.75,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {naive_NELLY_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.95,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {dwm_NELLY_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.update_traces(
    marker=dict(
        size=3,
        line=dict(width=2, color="DarkSlateGrey")
    )
)
fig.update_layout(
    font=dict(size=16),
    showlegend=False
)
fig.update_yaxes(range=[-0.5, 1])
fig.update_xaxes(tickangle=-90)

fig

In [36]:
# fig.write_image("slides/figures_regression/benchmark/pancancer_CV_NBS_cells_fixed-drug.pdf")

In [32]:
_, p = wilcoxon(
    naive_NELLY_fixed_drug.T["naive-NELLY"],
    screendl_fixed_drug.T["ScreenDL"], alternative="two-sided")
p

np.float64(0.001953125)

In [33]:
_, p = wilcoxon(
    dwm_NELLY_fixed_drug.T["DWM-NELLY"],
    screendl_fixed_drug.T["ScreenDL"], alternative="two-sided")
p

np.float64(0.001953125)

In [37]:
_, p = wilcoxon(
    screendl_fixed_drug.T["ScreenDL"],
    hidra_fixed_drug.T["HiDRA"], alternative="two-sided")
p

np.float64(0.001953125)

#### Precision@Q25

In [58]:
screendl_fixed_drug = pd.read_csv(
    "benchmarking/screendl/predictions_pancancer_NBS_cells/precision_fixed-drug_CV.csv",
    header=0,
    index_col=0
)

In [59]:
idx = screendl_fixed_drug.columns

In [60]:
screendl_fixed_drug = screendl_fixed_drug.set_axis(
    labels=["0","1","2","3","4","5","6","7","8","9"],
    axis=1
)

In [61]:
hidra_fixed_drug = pd.read_csv(
    "benchmarking/hidra/CV/predictions_pancancer_NBS_cells/precision_fixed-drug_CV.csv",
    index_col=0
)

In [62]:
hidra_fixed_drug.index = ["HiDRA"]

In [63]:
paccmann_fixed_drug = pd.read_csv(
    "benchmarking/paccmann_predictor/CV/predictions_pancancer_NBS_cells/precision_fixed-drug_CV.csv",
    index_col=0
)

In [64]:
naive_NELLY_fixed_drug = pd.read_csv(
    "naive_predictor/L1000_pancancer_NBS_cells/precision_fixed-drug_CV.csv",
    index_col=0
)

In [65]:
naive_NELLY_fixed_drug.index = ["naive-NELLY"]

In [66]:
dwm_NELLY_fixed_drug = pd.read_csv(
    "cDWM/L1000_pancancer_NBS_cells/precision_fixed-drug_CV.csv",
    index_col=0
)

In [67]:
dwm_NELLY_fixed_drug.index = ["DWM-NELLY"]

In [68]:
metrics_fixed_drug = pd.concat(
    [
        hidra_fixed_drug,
        paccmann_fixed_drug,
        screendl_fixed_drug,
        naive_NELLY_fixed_drug,
        dwm_NELLY_fixed_drug
    ]
).T

In [69]:
metrics_fixed_drug

,HiDRA,Paccmann,ScreenDL,naive-NELLY,DWM-NELLY
0,0.478261,0.478261,0.521739,0.521739,0.652174
1,0.478261,0.444444,0.500000,0.523810,0.652174
2,0.458333,0.521739,0.500000,0.500000,0.636364
3,0.478261,0.529412,0.521739,0.565217,0.666667
4,0.500000,0.478261,0.500000,0.521739,0.650000
5,0.454545,0.500000,0.500000,0.500000,0.636364
6,0.458333,0.454545,0.500000,0.500000,0.636364
7,0.500000,0.555556,0.545455,0.590909,0.666667
8,0.478261,0.521739,0.500000,0.536585,0.652174
9,0.416667,0.368421,0.454545,0.500000,0.500000


In [70]:
metrics_fixed_drug.index = idx

In [71]:
metrics_fixed_drug.reset_index(inplace=True)

In [72]:
metrics_fixed_drug = metrics_fixed_drug.melt(
    id_vars="index",
    var_name="Method",
    value_name="MPrecision"
)

In [73]:
metrics_fixed_drug.head()

,index,Method,MPrecision
0,fold_0,HiDRA,0.478261
1,fold_1,HiDRA,0.478261
2,fold_2,HiDRA,0.458333
3,fold_3,HiDRA,0.478261
4,fold_4,HiDRA,0.500000


In [74]:

palette = {
    "Paccmann": "#1F77B4",       # blue
    "HiDRA": "#FF7F0E",  # orange
    "ScreenDL": "#2CA02C",     # green
    "naive-NELLY": "#D62728",   # red
    "DWM-NELLY" : "#b38ce3"
}

fig = px.violin(metrics_fixed_drug,
                x="Method",
                y="MPrecision",
                color="Method",
                height=500,
                width=700,
                template="simple_white",
                box=True,
                points="all",
                labels={"Method": "",
                        "MPrecision": "Precision@Q25"
                       },
                color_discrete_map=palette,
                title= "CV on NBS cells | Fixed-drug precision@Q25" 
               )


fig.update_traces(
    jitter=0.05,
    pointpos=0,
    width=0.5
)

fig.add_annotation(
    x=0.03,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {hidra_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.add_annotation(
    x=0.23,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {paccmann_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.add_annotation(
    x=0.48,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {screendl_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.75,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {naive_NELLY_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.95,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {dwm_NELLY_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.update_traces(
    marker=dict(
        size=3,
        line=dict(width=2, color="DarkSlateGrey")
    )
)
fig.update_layout(
    font=dict(size=16),
    showlegend=False
)
fig.update_yaxes(range=[-0.5, 1])
fig.update_xaxes(tickangle=-90)

fig

In [75]:
# fig.write_image("slides/figures_regression/benchmark/pancancer_precision_CV_NBS_cells_fixed-drug.pdf")

In [ ]:
_, p = wilcoxon(
    naive_NELLY_fixed_drug.T["naive-NELLY"],
    screendl_fixed_drug.T["ScreenDL"], alternative="greater")
p

In [49]:
_, p = wilcoxon(
    dwm_NELLY_fixed_drug.T["DWM-NELLY"],
    screendl_fixed_drug.T["ScreenDL"], alternative="two-sided")
p

np.float64(0.001953125)

In [ ]:
_, p = wilcoxon(
    dwm_NELLY_fixed_drug.T["DWM-NELLY"],
    naive_NELLY_fixed_drug.T["naive-NELLY"], alternative="greater")
p

#### NDCG@Q25

In [76]:
screendl_fixed_drug = pd.read_csv(
    "benchmarking/screendl/predictions_pancancer_NBS_cells/ndcg_fixed-drug_CV.csv",
    header=0,
    index_col=0
)

In [77]:
idx = screendl_fixed_drug.columns

In [78]:
screendl_fixed_drug = screendl_fixed_drug.set_axis(
    labels=["0","1","2","3","4","5","6","7","8","9"],
    axis=1
)

In [79]:
hidra_fixed_drug = pd.read_csv(
    "benchmarking/hidra/CV/predictions_pancancer_NBS_cells/ndcg_fixed-drug_CV.csv",
    index_col=0
)

In [80]:
paccmann_fixed_drug = pd.read_csv(
    "benchmarking/paccmann_predictor/CV/predictions_pancancer_NBS_cells/ndcg_fixed-drug_CV.csv",
    index_col=0
)

In [81]:
naive_NELLY_fixed_drug = pd.read_csv(
    "naive_predictor/L1000_pancancer_NBS_cells/ndgc_fixed-drug_CV.csv",
    index_col=0
)

In [82]:
dwm_NELLY_fixed_drug = pd.read_csv(
    "cDWM/L1000_pancancer_NBS_cells/ndgc_fixed-drug_CV.csv",
    index_col=0
)

In [83]:
metrics_fixed_drug = pd.concat(
    [
        hidra_fixed_drug,
        paccmann_fixed_drug,
        naive_NELLY_fixed_drug,
        screendl_fixed_drug,
        dwm_NELLY_fixed_drug
    ]
).T

In [84]:
metrics_fixed_drug

,HiDRA,Paccmann,naive-NELLY,ScreenDL,DWM-NELLY
0,0.0,0.268795,0.435757,0.521833,0.701374
1,0.0,0.277148,0.471821,0.509153,0.694064
2,0.0,0.417310,0.400387,0.488358,0.675998
3,0.0,0.442117,0.487883,0.509057,0.747532
4,0.0,0.344707,0.420864,0.490524,0.711537
5,0.0,0.301362,0.372262,0.500556,0.684600
6,0.0,0.328657,0.405214,0.572676,0.709389
7,0.0,0.458292,0.495934,0.527477,0.735121
8,0.0,0.360554,0.407975,0.489047,0.643934
9,0.0,0.209433,0.384313,0.449285,0.413126


In [105]:
metrics_fixed_drug.index = idx

In [85]:
metrics_fixed_drug.reset_index(inplace=True)

In [86]:
metrics_fixed_drug = metrics_fixed_drug.melt(
    id_vars="index",
    var_name="Method",
    value_name="MNDCG"
)

In [87]:
metrics_fixed_drug.head()

,index,Method,MNDCG
0,0,HiDRA,0.0
1,1,HiDRA,0.0
2,2,HiDRA,0.0
3,3,HiDRA,0.0
4,4,HiDRA,0.0


In [88]:

palette = {
    "Paccmann": "#1F77B4",       # blue
    "HiDRA": "#FF7F0E",  # orange
    "ScreenDL": "#2CA02C",     # green
    "naive-NELLY": "#D62728",   # red
    "DWM-NELLY" : "#b38ce3"
}

fig = px.violin(metrics_fixed_drug,
                x="Method",
                y="MNDCG",
                color="Method",
                height=500,
                width=700,
                template="simple_white",
                box=True,
                points="all",
                labels={"Method": "",
                        "MNDCG": "NDCG@Q25"
                       },
                color_discrete_map=palette,
                title= "CV on NBS cells | Fixed-drug NDCG@Q25" 
               )


fig.update_traces(
    jitter=0.05,
    pointpos=0,
    width=0.5
)

fig.add_annotation(
    x=0.03,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {hidra_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.add_annotation(
    x=0.23,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {paccmann_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.48,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {naive_NELLY_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.75,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {screendl_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.add_annotation(
    x=0.95,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.95,
    text=f"M = {dwm_NELLY_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.update_traces(
    marker=dict(
        size=3,
        line=dict(width=2, color="DarkSlateGrey")
    )
)
fig.update_layout(
    font=dict(size=16),
    showlegend=False
)
fig.update_yaxes(range=[-0.5, 1])
fig.update_xaxes(tickangle=-90)

fig

In [89]:
# fig.write_image("slides/figures_regression/benchmark/pancancer_NDCG_CV_NBS_cells_fixed-drug.pdf")

In [ ]:
_, p = wilcoxon(
    naive_NELLY_fixed_drug.T["naive-NELLY"],
    screendl_fixed_drug.T["ScreenDL"], alternative="greater")
p

In [57]:
_, p = wilcoxon(
    dwm_NELLY_fixed_drug.T["DWM-NELLY"],
    screendl_fixed_drug.T["ScreenDL"], alternative="two-sided")
p

np.float64(0.00390625)

In [ ]:
_, p = wilcoxon(
    dwm_NELLY_fixed_drug.T["DWM-NELLY"],
    naive_NELLY_fixed_drug.T["naive-NELLY"], alternative="greater")
p

### NBS cells | Fixed-cell

#### PCC

In [90]:
screendl_fixed_cell = pd.read_csv(
    "benchmarking/screendl/predictions_pancancer_NBS_cells/msigdb_fixed-cell_evaluation.csv",
    header=0,
    index_col=0
)

In [91]:
screendl_fixed_cell.columns = ["0", "1", "2","3","4","5","6","7","8","9"]

In [92]:
hidra_fixed_cell = pd.read_csv(
    "benchmarking/hidra/CV/predictions_pancancer_NBS_cells/fixed-cell_evaluation_df.csv",
    index_col=0
)

In [93]:
hidra_fixed_cell.index = ["HiDRA"]

In [94]:
paccmann_fixed_cell = pd.read_csv(
    "benchmarking/paccmann_predictor/CV/predictions_pancancer_NBS_cells/fixed-cell_evaluation_df.csv",
    index_col=0
)

In [95]:
naive_NELLY_fixed_cell = pd.read_csv(
    "naive_predictor/L1000_pancancer_NBS_cells/fixed-cell_CV.csv",
    index_col=0
)

In [96]:
naive_NELLY_fixed_cell.index = ["naive-NELLY"]

In [97]:
dwm_NELLY_fixed_cell = pd.read_csv(
    "cDWM/L1000_pancancer_NBS_cells/fixed-cell_CV.csv",
    index_col=0
)

In [98]:
dwm_NELLY_fixed_cell.index = ["DWM-NELLY"]

In [46]:
metrics_fixed_cell = pd.concat(
    [
        screendl_fixed_cell, paccmann_fixed_cell, hidra_fixed_cell,
        naive_NELLY_fixed_cell, dwm_NELLY_fixed_cell
    ]
).T

In [47]:
metrics_fixed_cell.index = idx

In [48]:
metrics_fixed_cell.reset_index(inplace=True)

In [49]:
metrics_fixed_cell = metrics_fixed_cell.melt(
    id_vars="index",
    var_name="Method",
    value_name="MCorrelation"
)

In [50]:
metrics_fixed_cell.head()

,index,Method,MCorrelation
0,fold_0,ScreenDL,0.305253
1,fold_1,ScreenDL,0.333345
2,fold_2,ScreenDL,0.326688
3,fold_3,ScreenDL,0.323148
4,fold_4,ScreenDL,0.345700


In [54]:
# _, p = wilcoxon(per_cell_pcc_cv.T["CPV"], screendl_nbs_cell_fixed_cell.T["ScreenDL"], alternative="greater")

palette = {
    "Paccmann": "#1F77B4",       # blue
    "HiDRA": "#FF7F0E",  # orange
    "ScreenDL": "#2CA02C",     # green
    "naive-NELLY": "#D62728",   # red
    "DWM-NELLY" : "#b38ce3"
}

fig = px.violin(metrics_fixed_cell,
                x="Method",
                y="MCorrelation",
                color="Method",
                color_discrete_map=palette,
                height=500,
                width=700,
                template="simple_white",
                box=True,
                points="all",
                labels={"Method": "",
                        "MCorrelation": "Pearson Correlation"
                       },
                title= "CV on NBS cells | Fixed-cell Correlation"
                
                
         )



fig.add_annotation(
    x=0,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {screendl_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.2,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {paccmann_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.5,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {hidra_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.78,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {naive_NELLY_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.add_annotation(
    x=0.98,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {dwm_NELLY_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.update_traces(
    marker=dict(
        size=3,
        line=dict(width=2, color="DarkSlateGrey")
    )
)

fig.update_traces(
    jitter=0.05,
     pointpos=0,
    width=0.5
)

fig.update_layout(font=dict(size=16), showlegend=False)
fig.update_yaxes(range=[-0.5, 1])
fig.update_xaxes(tickangle=-90)
fig

In [55]:
# fig.write_image("slides/figures_regression/benchmark/pancancer_CV_NBS_cells_fixed-cell.pdf")

In [27]:
_, p = wilcoxon(
    naive_NELLY_fixed_cell.T["naive-NELLY"],
    hidra_fixed_cell.T["HiDRA"], alternative="greater")
p

np.float64(0.0009765625)

In [99]:
_, p = wilcoxon(
    dwm_NELLY_fixed_cell.T["DWM-NELLY"],
    hidra_fixed_cell.T["HiDRA"], alternative="two-sided")
p

np.float64(0.001953125)

In [29]:
_, p = wilcoxon(
    dwm_NELLY_fixed_cell.T["DWM-NELLY"],
    naive_NELLY_fixed_cell.T["naive-NELLY"], alternative="greater")
p

np.float64(0.0009765625)

#### Precision@Q25

In [100]:
screendl_fixed_cell = pd.read_csv(
    "benchmarking/screendl/predictions_pancancer_NBS_cells/precision_fixed-cell_CV.csv",
    header=0,
    index_col=0
)

In [101]:
screendl_fixed_cell.columns = ["0", "1", "2","3","4","5","6","7","8","9"]

In [102]:
hidra_fixed_cell = pd.read_csv(
    "benchmarking/hidra/CV/predictions_pancancer_NBS_cells/precision_fixed-cell_CV.csv",
    index_col=0
)

In [103]:
paccmann_fixed_cell = pd.read_csv(
    "benchmarking/paccmann_predictor/CV/predictions_pancancer_NBS_cells/precision_fixed-cell_CV.csv",
    index_col=0
)

In [104]:
naive_NELLY_fixed_cell = pd.read_csv(
    "naive_predictor/L1000_pancancer_NBS_cells/precision_fixed-cell_CV.csv",
    index_col=0
)

In [105]:
dwm_NELLY_fixed_cell = pd.read_csv(
    "cDWM/L1000_pancancer_NBS_cells/precision_fixed-cell_CV.csv",
    index_col=0
)

In [117]:
metrics_fixed_cell = pd.concat(
    [
        screendl_fixed_cell,
        paccmann_fixed_cell,
        hidra_fixed_cell,
        naive_NELLY_fixed_cell,
        dwm_NELLY_fixed_cell
    ]
).T

In [118]:
metrics_fixed_cell.index = idx

In [119]:
metrics_fixed_cell.reset_index(inplace=True)

In [120]:
metrics_fixed_cell = metrics_fixed_cell.melt(
    id_vars="index",
    var_name="Method",
    value_name="MPrecision"
)

In [121]:
metrics_fixed_cell.head()

,index,Method,MPrecision
0,fold_0,ScreenDL,0.384559
1,fold_1,ScreenDL,0.398413
2,fold_2,ScreenDL,0.400000
3,fold_3,ScreenDL,0.406250
4,fold_4,ScreenDL,0.393617


In [124]:
# _, p = wilcoxon(per_cell_pcc_cv.T["CPV"], screendl_nbs_cell_fixed_cell.T["ScreenDL"], alternative="greater")

palette = {
    "Paccmann": "#1F77B4",       # blue
    "HiDRA": "#FF7F0E",  # orange
    "ScreenDL": "#2CA02C",     # green
    "naive-NELLY": "#D62728",   # red
    "DWM-NELLY" : "#b38ce3"
}

fig = px.violin(metrics_fixed_cell,
                x="Method",
                y="MPrecision",
                color="Method",
                color_discrete_map=palette,
                height=500,
                width=700,
                template="simple_white",
                box=True,
                points="all",
                labels={"Method": "",
                        "MPrecision": "Precision@Q25"
                       },
                title= "CV on NBS cells | Fixed-cell precision@Q25"
                
                
         )



fig.add_annotation(
    x=0,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {screendl_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.2,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {paccmann_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.5,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {hidra_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.78,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {naive_NELLY_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.add_annotation(
    x=0.98,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {dwm_NELLY_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.update_traces(
    marker=dict(
        size=3,
        line=dict(width=2, color="DarkSlateGrey")
    )
)

fig.update_traces(
    jitter=0.05,
     pointpos=0,
    width=0.5
)

fig.update_layout(font=dict(size=16), showlegend=False)
fig.update_yaxes(range=[-0.5, 1])
fig.update_xaxes(tickangle=-90)
fig

In [125]:
# fig.write_image("slides/figures_regression/benchmark/pancancer_precision_CV_NBS_cells_fixed-cell.pdf")

In [ ]:
_, p = wilcoxon(
    naive_NELLY_fixed_cell.T["naive-NELLY"],
    hidra_fixed_cell.T["HiDRA"], alternative="greater")
p

In [106]:
_, p = wilcoxon(
    dwm_NELLY_fixed_cell.T["DWM-NELLY"],
    hidra_fixed_cell.T["HiDRA"], alternative="two-sided")
p

np.float64(0.001953125)

In [ ]:
_, p = wilcoxon(
    dwm_NELLY_fixed_cell.T["DWM-NELLY"],
    naive_NELLY_fixed_cell.T["naive-NELLY"], alternative="greater")
p

#### NDCG@Q25

In [107]:
screendl_fixed_cell = pd.read_csv(
    "benchmarking/screendl/predictions_pancancer_NBS_cells/ndcg_fixed-cell_CV.csv",
    header=0,
    index_col=0
)

In [108]:
screendl_fixed_cell.columns = ["0", "1", "2","3","4","5","6","7","8","9"]

In [109]:
hidra_fixed_cell = pd.read_csv(
    "benchmarking/hidra/CV/predictions_pancancer_NBS_cells/ndcg_fixed-cell_CV.csv",
    index_col=0
)

In [110]:
paccmann_fixed_cell = pd.read_csv(
    "benchmarking/paccmann_predictor/CV/predictions_pancancer_NBS_cells/ndcg_fixed-cell_CV.csv",
    index_col=0
)

In [111]:
naive_NELLY_fixed_cell = pd.read_csv(
    "naive_predictor/L1000_pancancer_NBS_cells/ndcg_fixed-cell_CV.csv",
    index_col=0
)

In [112]:
dwm_NELLY_fixed_cell = pd.read_csv(
    "cDWM/L1000_pancancer_NBS_cells/ndcg_fixed-cell_CV.csv",
    index_col=0
)

In [132]:
metrics_fixed_cell = pd.concat(
    [
        screendl_fixed_cell,
        paccmann_fixed_cell,
        hidra_fixed_cell,
        naive_NELLY_fixed_cell,
        dwm_NELLY_fixed_cell
    ]
).T

In [133]:
metrics_fixed_cell.index = idx

In [134]:
metrics_fixed_cell.reset_index(inplace=True)

In [135]:
metrics_fixed_cell = metrics_fixed_cell.melt(
    id_vars="index",
    var_name="Method",
    value_name="MNDCG"
)

In [136]:
metrics_fixed_cell.head()

,index,Method,MNDCG
0,fold_0,ScreenDL,0.320616
1,fold_1,ScreenDL,0.363592
2,fold_2,ScreenDL,0.331264
3,fold_3,ScreenDL,0.341920
4,fold_4,ScreenDL,0.332322


In [137]:
# _, p = wilcoxon(per_cell_pcc_cv.T["CPV"], screendl_nbs_cell_fixed_cell.T["ScreenDL"], alternative="greater")

palette = {
    "Paccmann": "#1F77B4",       # blue
    "HiDRA": "#FF7F0E",  # orange
    "ScreenDL": "#2CA02C",     # green
    "naive-NELLY": "#D62728",   # red
    "DWM-NELLY" : "#b38ce3"
}

fig = px.violin(metrics_fixed_cell,
                x="Method",
                y="MNDCG",
                color="Method",
                color_discrete_map=palette,
                height=500,
                width=700,
                template="simple_white",
                box=True,
                points="all",
                labels={"Method": "",
                        "MNDCG": "NDCG@Q25"
                       },
                title= "CV on NBS cells | Fixed-cell NDCG@Q25"
                
                
         )



fig.add_annotation(
    x=0,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {screendl_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.2,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {paccmann_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.5,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {hidra_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.add_annotation(
    x=0.78,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {naive_NELLY_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)


fig.add_annotation(
    x=0.98,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"M = {dwm_NELLY_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=12)
)

fig.update_traces(
    marker=dict(
        size=3,
        line=dict(width=2, color="DarkSlateGrey")
    )
)

fig.update_traces(
    jitter=0.05,
     pointpos=0,
    width=0.5
)

fig.update_layout(font=dict(size=16), showlegend=False)
fig.update_yaxes(range=[-0.5, 1])
fig.update_xaxes(tickangle=-90)
fig

In [138]:
# fig.write_image("slides/figures_regression/benchmark/pancancer_NDCG_CV_NBS_cells_fixed-cell.pdf")

In [ ]:
_, p = wilcoxon(
    naive_NELLY_fixed_cell.T["naive-NELLY"],
    hidra_fixed_cell.T["HiDRA"], alternative="greater")
p

In [113]:
_, p = wilcoxon(
    dwm_NELLY_fixed_cell.T["DWM-NELLY"],
    hidra_fixed_cell.T["HiDRA"], alternative="two-sided")
p

np.float64(0.001953125)

In [ ]:
_, p = wilcoxon(
    dwm_NELLY_fixed_cell.T["DWM-NELLY"],
    naive_NELLY_fixed_cell.T["naive-NELLY"], alternative="greater")
p

### NBS drugs | Fixed-drug

In [174]:
screendl_nbs_drug_fixed_drug = pd.read_csv("benchmarking/screendl/output/4k_CV_fixed_drug_NBS_drug.csv", header=0, index_col=0)

In [175]:
screendl_nbs_drug_fixed_drug.index = ["ScreenDL"]

In [176]:
_, p = wilcoxon(per_drug_pcc_cv.T["CPV"], screendl_nbs_drug_fixed_drug.T["ScreenDL"], alternative="greater")

In [177]:
p

0.0009765625

In [178]:
metrics_fixed_drug = pd.concat([screendl_nbs_drug_fixed_drug, per_drug_pcc_cv]).T

In [179]:
metrics_fixed_drug.reset_index(inplace=True)

In [180]:
metrics_fixed_drug = metrics_fixed_drug.melt(id_vars="index", var_name="Method", value_name="MCorrelation")

In [ ]:

_, p = wilcoxon(per_drug_pcc_cv.T["CPV"], screendl_nbs_drug_fixed_drug.T["ScreenDL"], alternative="greater")


fig = px.violin(metrics_fixed_drug,
                x="Method",
                y="MCorrelation",
                color="Method",
                height=700,
                width=700,
                template="simple_white",
                box=False,
                points=False,
                labels={"Method": "",
                        "MCorrelation": "Pearson Correlation"
                       },
                title= "CV on NBS drugs | Fixed-drug Correlation"
                
                
         )


# Step 2: Add paired-line traces manually
df_wide = metrics_fixed_drug.pivot(index="index", columns="Method", values="MCorrelation").reset_index()
for i, row in df_wide.iterrows():
    fig.add_trace(go.Scatter(
        x=["CPV", "ScreenDL"],
        y=[row["CPV"], row["ScreenDL"]],
        mode="lines+markers",
        line=dict(color="gray", width=1),
        marker=dict(size=6, color="black"),
        name=row["index"],
        showlegend=False  # hide fold labels
    ))

#fig.update_traces(jitter=0.05, pointpos=0)

fig.add_annotation(
    x=0.5,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"Wilcoxon signed-rank test, p-value = {p:.2e}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=14)
)

fig.add_annotation(
    x=0.1,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.8,
    text=f"Mean = {screendl_nbs_drug_fixed_drug.mean(1).item():.2f}, Median = {screendl_nbs_drug_fixed_drug.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=14)
)

fig.add_annotation(
    x=0.95,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.8,
    text=f"Mean = {per_drug_pcc_cv.mean(1).item():.2f}, Median = {per_drug_pcc_cv.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=14)
)

fig.update_yaxes(range=[0, 1])
fig

In [ ]:
fig.write_image("slides/figures_regression/standard_comparison_CV_NBS_drugs_fixed-drug.pdf")

### NBS drugs | Fixed-cell

In [ ]:
screendl_nbs_drug_fixed_cell = pd.read_csv("benchmarking/screendl/output/4k_CV_fixed_cell_NBS_drug.csv", header=0, index_col=0)

In [ ]:
screendl_nbs_drug_fixed_cell.index = ["ScreenDL"]

In [ ]:
screendl_nbs_drug_fixed_cell

In [ ]:
_, p = wilcoxon(per_cell_pcc_cv.T["CPV"], screendl_nbs_drug_fixed_cell.T["ScreenDL"], alternative="greater")

In [ ]:
p

In [189]:
metrics_fixed_drug = pd.concat([screendl_nbs_drug_fixed_cell, per_cell_pcc_cv]).T

In [190]:
metrics_fixed_drug.reset_index(inplace=True)

In [191]:
metrics_fixed_drug = metrics_fixed_drug.melt(id_vars="index", var_name="Method", value_name="MCorrelation")

In [ ]:

_, p = wilcoxon(per_cell_pcc_cv.T["CPV"], screendl_nbs_drug_fixed_cell.T["ScreenDL"], alternative="greater")


fig = px.violin(metrics_fixed_drug,
                x="Method",
                y="MCorrelation",
                color="Method",
                height=700,
                width=700,
                template="simple_white",
                box=False,
                points=False,
                labels={"Method": "",
                        "MCorrelation": "Pearson Correlation"
                       },
                title= "CV on NBS drugs | Fixed-cell Correlation"
                
                
         )


# Step 2: Add paired-line traces manually
df_wide = metrics_fixed_drug.pivot(index="index", columns="Method", values="MCorrelation").reset_index()
for i, row in df_wide.iterrows():
    fig.add_trace(go.Scatter(
        x=["CPV", "ScreenDL"],
        y=[row["CPV"], row["ScreenDL"]],
        mode="lines+markers",
        line=dict(color="gray", width=1),
        marker=dict(size=6, color="black"),
        name=row["index"],
        showlegend=False  # hide fold labels
    ))

#fig.update_traces(jitter=0.05, pointpos=0)

fig.add_annotation(
    x=0.5,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=1,
    text=f"Wilcoxon signed-rank test, p-value = {p:.2e}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=14)
)

fig.add_annotation(
    x=0.1,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.8,
    text=f"Mean = {screendl_nbs_drug_fixed_cell.mean(1).item():.2f}, Median = {screendl_nbs_drug_fixed_cell.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=14)
)

fig.add_annotation(
    x=0.95,  # Adjust if you have multiple violins (e.g., grouped by drug)
    y=0.8,
    text=f"Mean = {per_cell_pcc_cv.mean(1).item():.2f}, Median = {per_cell_pcc_cv.median(1).item():.2f}",
    xref="paper", yref="paper",
    showarrow=False,
    yanchor='bottom',
    font=dict(color="black", size=14)
)

fig.update_yaxes(range=[0, 1])

fig

fig.write_image("slides/figures_regression/standard_comparison_CV_NBS_drugs_fixed-cell.pdf")